In [3]:
#part-1 -Data Cleaning & Preparation

import pandas as pd
import numpy as np

# Loading Marketing Funnel
mql = pd.read_csv("olist_marketing_qualified_leads_dataset.csv")
closed = pd.read_csv("olist_closed_deals_dataset.csv")

# Loading Brazilian E-commerce datasets
sellers = pd.read_csv("olist_sellers_dataset.csv")
orders = pd.read_csv("olist_orders_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")

# -----------------------
# Cleaning Function
# -----------------------
def clean_df(df, df_name):
    print(f"\nCleaning {df_name}...")
    # Drop duplicates
    df = df.drop_duplicates()
    # Standardize column names
    df.columns = df.columns.str.strip().str.lower()
    # Trim only string/object columns that are actually strings
    for col in df.select_dtypes(include=["object"]):
        df[col] = df[col].astype(str).str.strip()  # safely strip spaces
        # Optional: replace 'nan' strings back to real NaN
        df[col] = df[col].replace("nan", np.nan)
    # Handle missing values
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print("Missing values:\n", missing[missing > 0])
    else:
        print("No missing values.")
    return df


# Cleaning all loaded datasets
mql = clean_df(mql, "MQL")
closed = clean_df(closed, "Closed Deals")
sellers = clean_df(sellers, "Sellers")
orders = clean_df(orders, "Orders")
order_items = clean_df(order_items, "Order Items")
payments = clean_df(payments, "Payments")

# Convert date columns to datetime
date_cols = ['first_contact_date', 'won_date', 'order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in date_cols:
    if col in mql.columns:
        mql[col] = pd.to_datetime(mql[col], errors='coerce')
    if col in closed.columns:
        closed[col] = pd.to_datetime(closed[col], errors='coerce')
    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col], errors='coerce')

# -----------------------
# Merge Step 1: Marketing Funnel
# -----------------------
funnel_df = pd.merge(mql, closed, on="mql_id", how="left")

# -----------------------
# Merge Step 2: Adding Seller Info
# -----------------------
funnel_df = pd.merge(funnel_df, sellers, on="seller_id", how="left")

# -----------------------
# Merge Step 3: Adding Order Items
# -----------------------
funnel_orders = pd.merge(funnel_df, order_items, on="seller_id", how="left")

# -----------------------
# Merge Step 4: Adding Orders
# -----------------------
funnel_orders = pd.merge(funnel_orders, orders, on="order_id", how="left")

# -----------------------
# Merge Step 5: Adding Payments
# -----------------------
funnel_orders = pd.merge(funnel_orders, payments, on="order_id", how="left")

print(f"Final merged dataset shape: {funnel_orders.shape}")



Cleaning MQL...
Missing values:
 origin    60
dtype: int64

Cleaning Closed Deals...
Missing values:
 business_segment                   1
lead_type                          6
lead_behaviour_profile           177
has_company                      779
has_gtin                         778
average_stock                    776
business_type                     10
declared_product_catalog_size    773
dtype: int64

Cleaning Sellers...
No missing values.

Cleaning Orders...
Missing values:
 order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

Cleaning Order Items...
No missing values.

Cleaning Payments...
No missing values.
Final merged dataset shape: (12888, 37)


In [5]:
#Part-2 - Data Analysis
#Conversion Rates
# Leads → MQL → SQL → Won
total_leads = len(mql)
total_mql = funnel_df["mql_id"].nunique()
total_won = funnel_df[~funnel_df["won_date"].isnull()]["mql_id"].nunique()

lead_to_mql = total_mql / total_leads * 100
mql_to_won = total_won / total_mql * 100

print(f"Lead → MQL conversion: {lead_to_mql:.2f}%")
print(f"MQL → Won conversion: {mql_to_won:.2f}%")


Lead → MQL conversion: 100.00%
MQL → Won conversion: 10.53%


In [7]:
#Top 3 Channels by ROI
# If cost per lead is available in closed deals
funnel_orders["revenue"] = funnel_orders["price"]  # from order_items
funnel_orders["cost"] = funnel_orders["declared_monthly_revenue"] / 10  # Example placeholder

funnel_orders["roi"] = (funnel_orders["revenue"] - funnel_orders["cost"]) / funnel_orders["cost"]

top_channels = funnel_orders.groupby("origin")["roi"].mean().nlargest(3)
print("Top 3 Channels by ROI:\n", top_channels)


Top 3 Channels by ROI:
 origin
direct_traffic    inf
display           inf
email             inf
Name: roi, dtype: float64


In [9]:
#Average Sales Cycle Length
funnel_orders["sales_cycle_days"] = (funnel_orders["won_date"] - funnel_orders["first_contact_date"]).dt.days
avg_cycle = funnel_orders["sales_cycle_days"].mean()
print(f"Average Sales Cycle Length: {avg_cycle:.2f} days")


Average Sales Cycle Length: 24.89 days


In [11]:
#Cleaned Dataset for Dashboard
funnel_orders.to_csv("final_marketing_sales_dataset.csv", index=False)
print("Dataset ready for Power BI.")


Dataset ready for Power BI.
